# Value Concentration

This notebook measures the concentration of market capitalization among the 200 largest operational enterprises in the local database.

The x-axis represents the ranking percentile, from the largest company to the weakest company in the Top 200. The y-axis represents cumulative market capitalization in USD millions.

The analysis follows the project conventions: data is read from the `enterprises` table, monetary values are stored in USD millions, and unknown or invalid values are excluded rather than estimated.

## 1. Configure the analysis environment

The notebook uses pandas for tabular transformations, NumPy for numeric checks, Matplotlib for export compatibility, and Plotly for the interactive concentration curve.

In [1]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

# Keep naming explicit so the transformations remain easy to audit.
TOP_N = 200
CAPITALIZATION_COLUMN = 'capitalization_millions'

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')


def find_database_path():
    """Locate database.db from the notebook directory or one of its parents."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for directory in candidates:
        database_path = directory / 'database.db'
        if database_path.exists():
            return database_path
    raise FileNotFoundError('database.db was not found from the current directory')


DB_PATH = find_database_path()
EXPORT_DIR = DB_PATH.parent / 'analyses' / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Database: {DB_PATH}')
print(f'Top-N target: {TOP_N}')

Database: c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Top-N target: 200


## 2. Load the data and validate the schema

The source is the operational `enterprises` table. The required fields are the enterprise identifier, the enterprise name, and numeric market capitalization. Missing or non-positive capitalizations are excluded because they cannot contribute to a meaningful ranking.

In [2]:
required_columns = {'id', 'name', 'capitalization'}

with sqlite3.connect(DB_PATH) as connection:
    enterprise_columns = {
        row[1] for row in connection.execute('PRAGMA table_info(enterprises)').fetchall()
    }
    missing_columns = required_columns - enterprise_columns
    if missing_columns:
        raise ValueError(f'Missing required enterprises columns: {sorted(missing_columns)}')

    raw_enterprises = pd.read_sql_query(
        'SELECT id, name, capitalization FROM enterprises',
        connection
    )

# Convert the database field to the project-wide numeric USD-million convention.
raw_enterprises[CAPITALIZATION_COLUMN] = pd.to_numeric(
    raw_enterprises['capitalization'],
    errors='coerce'
)

quality_summary = pd.Series({
    'raw_rows': len(raw_enterprises),
    'missing_capitalization': raw_enterprises[CAPITALIZATION_COLUMN].isna().sum(),
    'non_positive_capitalization': (raw_enterprises[CAPITALIZATION_COLUMN] <= 0).sum(),
    'duplicate_ids': raw_enterprises['id'].duplicated().sum(),
    'duplicate_names': raw_enterprises['name'].astype('string').str.strip().str.lower().duplicated().sum(),
})
display(quality_summary.to_frame('count'))

# Keep one row per enterprise id, preferring the first source record when duplicates exist.
enterprises = (
    raw_enterprises
    .drop_duplicates(subset=['id'], keep='first')
    .dropna(subset=['name', CAPITALIZATION_COLUMN])
    .loc[lambda frame: frame[CAPITALIZATION_COLUMN] > 0]
    .copy()
)

if len(enterprises) < TOP_N:
    raise ValueError(f'Only {len(enterprises)} valid enterprises are available; {TOP_N} are required.')

print(f'Valid enterprises available for ranking: {len(enterprises):,}')

,count
raw_rows,2557
missing_capitalization,2271
non_positive_capitalization,0
duplicate_ids,0
duplicate_names,1


Valid enterprises available for ranking: 286


## 3. Prepare the Top 200 by market capitalization

Enterprises are sorted from highest to lowest capitalization. The ranking is an integer from 1 to 200 and is independent of the database identifier.

In [3]:
top200 = (
    enterprises
    .sort_values(
        by=[CAPITALIZATION_COLUMN, 'name', 'id'],
        ascending=[False, True, True],
        kind='mergesort'
    )
    .head(TOP_N)
    .reset_index(drop=True)
)
top200['rank'] = np.arange(1, TOP_N + 1)

assert len(top200) == TOP_N
assert top200['rank'].tolist() == list(range(1, TOP_N + 1))
assert top200[CAPITALIZATION_COLUMN].is_monotonic_decreasing

display(top200.head(10)[['rank', 'id', 'name', CAPITALIZATION_COLUMN]])

,rank,id,name,capitalization_millions
0,1,17,Nvidia,"5,470,000.00"
1,2,4,Google,"4,560,000.00"
2,3,158,Apple,"4,500,000.00"
3,4,5,Microsoft,"3,450,000.00"
4,5,1615,Amazon,"2,900,000.00"
5,6,1967,TSMC,"2,170,000.00"
6,7,53,Broadcom,"1,835,000.00"
7,8,1668,Meta,"1,503,000.00"
8,9,159,SpaceX,"1,480,000.00"
9,10,9,AWS,"1,200,000.00"


## 4. Calculate ranking percentiles and cumulative value

For rank `r`, the percentile is `rank_pct = r / 200 * 100`. The cumulative series follows the same descending order and is accompanied by its share of the Top 200 total.

In [4]:
top200['rank_pct'] = top200['rank'] / TOP_N * 100
top200['cumulative_capitalization_millions'] = top200[CAPITALIZATION_COLUMN].cumsum()
total_top200 = top200[CAPITALIZATION_COLUMN].sum()
top200['cumulative_share_pct'] = (
    top200['cumulative_capitalization_millions'] / total_top200 * 100
)

# The percentile starts at 0% for the first company so the curve has a natural origin.
curve = pd.concat([
    pd.DataFrame({
        'rank': [0],
        'rank_pct': [0.0],
        'cumulative_capitalization_millions': [0.0],
        'cumulative_share_pct': [0.0],
    }),
    top200[['rank', 'rank_pct', 'cumulative_capitalization_millions', 'cumulative_share_pct']]
], ignore_index=True)

assert curve['rank_pct'].is_monotonic_increasing
assert curve['cumulative_capitalization_millions'].is_monotonic_increasing
assert np.isclose(curve.iloc[-1]['cumulative_capitalization_millions'], total_top200)

display(top200[['rank', 'rank_pct', 'name', CAPITALIZATION_COLUMN, 'cumulative_capitalization_millions', 'cumulative_share_pct']].head(10))

,rank,rank_pct,name,capitalization_millions,cumulative_capitalization_millions,cumulative_share_pct
0,1,0.50,Nvidia,"5,470,000.00","5,470,000.00",14.50
1,2,1.00,Google,"4,560,000.00","10,030,000.00",26.59
2,3,1.50,Apple,"4,500,000.00","14,530,000.00",38.51
3,4,2.00,Microsoft,"3,450,000.00","17,980,000.00",47.66
4,5,2.50,Amazon,"2,900,000.00","20,880,000.00",55.34
5,6,3.00,TSMC,"2,170,000.00","23,050,000.00",61.10
6,7,3.50,Broadcom,"1,835,000.00","24,885,000.00",65.96
7,8,4.00,Meta,"1,503,000.00","26,388,000.00",69.94
8,9,4.50,SpaceX,"1,480,000.00","27,868,000.00",73.87
9,10,5.00,AWS,"1,200,000.00","29,068,000.00",77.05


## 5. Plot the value concentration curve

The curve shows how quickly the Top 200 capitalization total accumulates as lower-ranked companies are added. Key percentile points are annotated directly on the chart.

In [10]:
def format_usd_millions(value):
    """Format a USD-million amount compactly for chart annotations."""
    if value >= 1_000_000:
        return f'${value / 1_000_000:.2f}T'
    if value >= 1_000:
        return f'${value / 1_000:.1f}B'
    return f'${value:,.0f}M'


key_percentiles = [10, 25, 50, 100]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=curve['rank_pct'],
    y=curve['cumulative_share_pct'],
    mode='lines',
    name='Cumulative share of Top 200 capitalization',
    line={'color': '#516B0E', 'width': 3},
    customdata=curve[['rank', 'cumulative_capitalization_millions']].to_numpy(),
    hovertemplate=(
        '<b>Rank percentile:</b> %{x:.1f}%<br>'
        '<b>Companies included:</b> %{customdata[0]:.0f}<br>'
        '<b>Cumulative share of Top 200:</b> %{y:.1f}%<br>'
        '<b>Cumulative capitalization:</b> %{customdata[1]:,.0f} M$<extra></extra>'
    )
))

for percentile in key_percentiles:
    row = curve.iloc[(curve['rank_pct'] - percentile).abs().argmin()]
    fig.add_annotation(
        x=row['rank_pct'],
        y=row['cumulative_share_pct'],
        text=f"{percentile}%: {row['cumulative_share_pct']:.1f}% ({format_usd_millions(row['cumulative_capitalization_millions'])})",
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=-38 if percentile < 100 else 35,
        bgcolor='rgba(253,250,244,0.92)',
        bordercolor='#D5CDC5',
        borderwidth=1,
        font={'size': 10, 'color': '#28241E'}
    )

fig.update_layout(
    title={
        'text': 'Value Concentration: Cumulative Share of Top 200 Market Capitalization',
        'x': 0.03,
        'xanchor': 'left',
        'font': {'size': 21, 'color': '#28241E'}
    },
    xaxis={
        'title': 'Enterprise ranking percentile (largest to smallest)',
        'range': [0, 100],
        'ticksuffix': '%',
        'dtick': 10,
        'gridcolor': '#EEE9E1',
        'zeroline': False
    },
    yaxis={
        'title': 'Cumulative share of Top 200 capitalization',
        'range': [0, 100],
        'ticksuffix': '%',
        'dtick': 10,
        'gridcolor': '#E4DED5',
        'zeroline': False
    },
    template='plotly_white',
    hovermode='x unified',
    showlegend=False,
    margin={'l': 90, 'r': 35, 't': 100, 'b': 80},
    height=650,
    paper_bgcolor='#FDFAF4',
    plot_bgcolor='#FDFAF4'
)

fig.show()

In [9]:
# Explicit notebook renderer for environments that do not auto-render Plotly MIME output.
fig.show(renderer='notebook_connected')

## 6. Add controls and export the results

Assertions verify the expected Top 200 size, ranking order, cumulative monotonicity, and total consistency. The transformed dataset and an embeddable HTML figure are exported for reuse.

In [6]:
# Report the concentration captured by several standard cutoffs.
cutoff_ranks = [1, 5, 10, 20, 50, TOP_N]
concentration_summary = pd.DataFrame({
    'cutoff_rank': cutoff_ranks,
    'rank_pct': [rank / TOP_N * 100 for rank in cutoff_ranks],
    'cumulative_capitalization_millions': [
        top200.loc[top200['rank'] == rank, 'cumulative_capitalization_millions'].iloc[0]
        for rank in cutoff_ranks
    ],
    'share_of_top200_pct': [
        top200.loc[top200['rank'] == rank, 'cumulative_share_pct'].iloc[0]
        for rank in cutoff_ranks
    ],
})

assert len(top200) == TOP_N
assert top200['rank'].is_unique
assert top200['rank'].min() == 1
assert top200['rank'].max() == TOP_N
assert top200['cumulative_capitalization_millions'].is_monotonic_increasing
assert np.isclose(
    top200['cumulative_capitalization_millions'].iloc[-1],
    top200[CAPITALIZATION_COLUMN].sum()
)
assert np.isclose(top200['cumulative_share_pct'].iloc[-1], 100.0)

display(concentration_summary)

csv_path = EXPORT_DIR / 'value_concentration_top200.csv'
html_path = EXPORT_DIR / 'value_concentration_top200.html'

top200.to_csv(csv_path, index=False, encoding='utf-8')
fig.write_html(html_path, include_plotlyjs='cdn')

print(f'Exported transformed data: {csv_path}')
print(f'Exported interactive figure: {html_path}')

,cutoff_rank,rank_pct,cumulative_capitalization_millions,share_of_top200_pct
0,1,0.50,"5,470,000.00",14.50
1,5,2.50,"20,880,000.00",55.34
2,10,5.00,"29,068,000.00",77.05
3,20,10.00,"34,353,770.00",91.06
4,50,25.00,"37,123,660.00",98.40
5,200,100.00,"37,727,864.00",100.00


Exported transformed data: c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\value_concentration_top200.csv
Exported interactive figure: c:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\value_concentration_top200.html
